In [79]:
import pandas as pd
import numpy as np
import h5py
import subprocess
import datetime

from pathlib import Path
from tqdm.notebook import tqdm

from mobgap.data import GenericMobilisedDataset
from mobgap.pipeline import MobilisedPipelineImpaired

In [80]:
# Experimental Parameters

# which files to find
root_dir = r"X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\\"
pattern = f"*6MWT.h5"
desired_sensor = "Lumbar"



In [100]:
# Functions
def find_files_with_progress(root_dir, pattern):
    root_dir = Path(root_dir)
    cmd = [
        "powershell",
        "-Command",
        f"Get-ChildItem -Path '{root_dir}' -Recurse -File -Include '{pattern}' | ForEach-Object {{ $_.FullName }}"
    ]

    files = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)

    pbar = tqdm(desc=f"Scanning {root_dir}", unit="files", dynamic_ncols=True)
    try:
        for line in process.stdout:
            line = line.strip()
            if line:
                files.append(str(Path(line)))
                pbar.update(1)
    finally:
        process.stdout.close()
        process.wait()
        pbar.close()

    print(f"Found {len(files)} matching files.")
    return files

def get_acc_gyr_from_h5(path, desired_sensor, prints = False):
    with h5py.File(path, 'r') as file:
        if prints:
            print()
            attr_names = list(file.attrs.keys())
            print(f"Attribute names: {attr_names}")
            top_level_keys = list(file)
            print(f"    MonitorLabelList: {[x.decode('UTF-8') for x in list(file.attrs["MonitorLabelList"])]}")
            print(f"    CaseIdList: {[x.decode('UTF-8') for x in list(file.attrs["CaseIdList"])]}")

        # get the sensor ID from the label list, assumes element i in MonitorLabelList is the label for element i in CaseIdList.
        desired_sensor_index = [x.decode('UTF-8') for x in list(file.attrs["MonitorLabelList"])].index(desired_sensor)
        desired_sensor_id = list(file.attrs["CaseIdList"])[desired_sensor_index].decode('UTF-8')

        if prints:
            print(f"        Lumbar Sensor ID: {desired_sensor_id}")
            SI_level_keys = list(file[desired_sensor_id])
            cal_level_keys = list(file[desired_sensor_id]["Calibrated"])
            raw_level_keys = list(file[desired_sensor_id]["Raw"])
    
            print("Each SI contains:", SI_level_keys)
            print("    Calibrated contains:", cal_level_keys)
            #print("    Raw contains:", raw_level_keys)

        acc_data = np.array(file[desired_sensor_id]["Calibrated"]["Accelerometers"])
        gyr_data = np.array(file[desired_sensor_id]["Calibrated"]["Gyroscopes"])
        time_data = np.array(file[desired_sensor_id]["Time"])

        if prints:
            print("        Shape of calibrated acc data:", np.shape(acc_data))
            print("        Shape of calibrated gyr data:", np.shape(gyr_data))
            print("        Shape of time", np.shape(time_data))
            print()
    
    if len(acc_data) == len(gyr_data) == len(time_data):
        time_data_reshaped = np.reshape(time_data, (-1, 1))
        acc_gyr_data = np.hstack((time_data_reshaped, acc_data, gyr_data))
        acc_gyr_data_df = pd.DataFrame(data = acc_gyr_data, columns = ["Time", "Acc_X", "Acc_Y", "Acc_Z", "Gyr_X", "Gyr_Y", "Gyr_Z"])

    if prints:
        print(acc_gyr_data_df.head(5))
        print()
        
    return acc_gyr_data_df

def save_to_mat_files(acc_gyr, subject_metadata, prints = False):
    data = {"TimeMeasure1": {}}
    
    if prints:
        print(f"NaNs (before interpolation) = {acc_gyr.isna().sum().sum()}")
    
    # get sample rate
    time_seconds = acc_gyr["Time"].astype('int64') / 1e6 # microseconds to seconds
    duration = time_seconds.iloc[-1] - time_seconds.iloc[0]
    if duration <= 0:
        raise ValueError("Invalid or zero-duration time range in acc_gyr.")
    fs = int(round(len(acc_gyr) / duration))

    # Convert timestamps to seconds for MATLAB
    reformatted_time = time_seconds.values.reshape(-1, 1)

    sensors = {
        "LowerBack": {
            "Fs": {"Acc": fs, "Gyr": fs},
            "Acc": acc_gyr[["Acc_X", "Acc_Y", "Acc_Z"]].values,
            "Gyr": acc_gyr[["Gyr_X", "Gyr_Y", "Gyr_Z"]].values,
            "Timestamp": reformatted_time,
        }
    }

    data["TimeMeasure1"]["SixMWT"] = {
        "SU": sensors,
        "StartDateTime": datetime.datetime.fromtimestamp(time_seconds[0]).strftime("%d-%b-%Y %H:%M:%S"),
        "TimeZone": "Europe/UK",
        "TrialName": "Trial 1",
    }

    if prints:
        print(data)
    
    # create infoForAlgo struct using metadata
    infoForAlgo = {
        "TimeMeasure1": {
            "Subject_ID": subject_metadata["SubjectNo"],
            "Cohort": subject_metadata["Cohort"],
            "Gender": subject_metadata["NewGender"],
            "Handedness": subject_metadata["HandPreference"],
            "Age": subject_metadata["NewAge"],
            "Weight": subject_metadata["Weight"],
            "Height": subject_metadata["Height"],
            #"SensorHeight": subject_metadata["senheight"],
            "SensorType_SU": "OPAL",
            "SensorAttachment_SU": "Body-Worn",
        }
    }

    if prints:
        print(infoForAlgo)

In [16]:
# Get all file paths (only needs to be run once so in own cell)
file_paths = find_files_with_progress(root_dir, pattern)
subject_ids = [path.split("\\")[-1].split("_")[0] for path in file_paths]

Scanning X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829): 0files [00:00, ?files/s]

Found 204 matching files.


In [101]:
metadata_file_path = root_dir + "Sheffield_MS-SMART_Maindataset.xlsx"
metadata_2_file_path = root_dir + "LongitudinalStudy_MS-SMART.xlsx"
metadata = pd.read_excel(metadata_file_path)
metadata_2 = pd.read_excel(metadata_2_file_path)

display(metadata)

for path in file_paths:
    print(path)
    subject_id = path.split("\\")[-3]
    acc_gyr_data = get_acc_gyr_from_h5(path, desired_sensor)

    metadata_row = metadata[metadata["SubjectNo"] == subject_id]
    save_to_mat_files(acc_gyr_data, metadata_row, prints = False)
    bruh

,SubjectNo,HandPreference,Height,Weight,NewAge,NewGender,NewCentre,RaceID,EDSSAtRandomisation,AllocatedTreatment,...,T2VolBase,BVBase,BV24,BV96,DGMBase,CGMBase,DGM96,CGM96,PDGVC96,PCGVC96
0,20001,2,2,82,45.83,1,20,1,4.5,3,...,2300,1473793,1547482.65,943227.52,47303,827778,47953.0,842757.0,1.37,1.81
1,20002,2,2,63,53.68,2,20,1,5.5,1,...,1973,1411222,-508039.92,155234.42,43651,791304,43145.0,774353.0,-1.16,-2.14
2,20003,1,2,64,64.62,1,20,1,6.5,4,...,7131,1449581,666807.26,463865.92,46157,813964,45198.0,799048.0,-2.08,-1.83
3,20004,2,1,84,46.02,2,20,1,5.5,2,...,79,1574418,1747603.98,1291022.76,49967,844240,50337.0,844391.0,0.74,0.02
4,20005,2,2,65,63.69,2,20,1,4.0,4,...,14643,1469083,1733517.94,-602324.03,44634,789934,44328.0,784470.0,-0.69,-0.69
5,20006,2,2,65,61.41,2,20,1,5.5,1,...,6630,1346740,983120.20,-377087.20,40326,778870,40553.0,766954.0,0.56,-1.53
6,20007,2,2,116,64.16,2,20,1,6.0,2,...,14850,1400675,-308148.50,966465.75,41278,787865,40765.0,784839.0,-1.24,-0.38
7,20009,1,2,96,60.03,1,20,1,6.5,4,...,2462,1413348,-240269.16,-1370947.56,43545,790643,44745.0,803555.0,2.76,1.63
8,20010,2,2,85,60.64,2,20,1,5.5,3,...,671,1464341,834674.37,1478984.41,45431,828969,NaN,NaN,NaN,NaN
9,20013,2,2,70,62.40,1,20,1,4.5,2,...,19073,1418542,56741.68,-1461098.26,38864,806804,38811.0,804514.0,-0.14,-0.28


X:\insigneo_brc1\GroupFiles\Study_Data\MS_SMART (STH17249; STH18829)\Healthy controls\C1\Baseline_week0\C1_20161019_101837_OPAL_6MWT.h5


KeyError: 'Cohort'